<a href="https://colab.research.google.com/github/mayait/CursoAnalisisDatos_IA_2026/blob/main/sitio/labs/lab_14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Laboratorio 14 · Modelos de propensión: clasificación

Las dos semanas anteriores predijiste un número: cuántas unidades, cuántos dólares. Hoy predices
**quién**. La variable objetivo ya no es continua sino un sí o un no —este cliente abandona o no
abandona— y con eso cambia todo lo demás: la métrica, la línea base, la forma de equivocarse y, sobre
todo, para qué sirve el resultado.

Y aquí está la idea que hay que llevarse: **un modelo de clasificación no sirve para acertar.** Sirve
para **ordenar la cartera** y decidir a quién llama el equipo comercial el lunes con el presupuesto que
hay. Un modelo con un 73 % de exactitud puede valer mucho dinero y otro con un 98 % puede no valer
nada. Al final del cuaderno vas a poder decir cuál es cuál sin dudar.

> **Hoy haces** · Construyes la variable de abandono de Comercial Andina con la ventana de observación
> separada de la ventana de resultado para no hacer trampa (90 min). Ajustas una regresión logística y
> un árbol de decisión y los comparas por validación cruzada estratificada contra `DummyClassifier`. Le
> pones precio en dólares a cada celda de la matriz de confusión y calculas el retorno de la campaña.
> Mueves el umbral y construyes la curva de ganancia: si el presupuesto alcanza para llamar al 20 % de
> la cartera, decides a quién. Cierras auditando un fragmento de código de IA con desempeño perfecto.
>
> **Entrega** · Este cuaderno ejecutado, la recomendación de focalización del caso del grupo —a
> quiénes, con qué umbral, con qué retorno esperado y con los supuestos de dinero escritos—, la matriz
> de confusión traducida a dólares, y la corrección del fragmento de la IA con la cifra de cuánto
> exageraba. Nombre de archivo: `lab_14_apellido.ipynb`.

In [ ]:
# --- Setup del entorno ---
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# Los datos de Comercial Andina viven en sitio/datos/
REPO = "https://github.com/mayait/CursoAnalisisDatos_IA_2026.git"
COPIA = Path("/content/CursoAnalisisDatos_IA_2026")
CANDIDATOS = [Path("../datos"), Path("datos"), Path("sitio/datos"),
              COPIA / "sitio" / "datos"]
DATOS = next((p for p in CANDIDATOS if p.exists()), None)
if DATOS is None:
    # En Colab el cuaderno llega solo: se trae el repositorio una sola vez.
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(COPIA)], check=True)
    DATOS = COPIA / "sitio" / "datos"

print("Setup completo ✓")
print(f"pandas {pd.__version__} · datos en {DATOS.resolve()}")

## 1. La variable objetivo no viene en el archivo: se define

En Comercial Andina nadie llama para darse de baja. Un cliente simplemente deja de aparecer. Así que
«abandono» no es un dato que se lea: es una **definición que alguien tiene que escribir y defender**, y
cambiarla cambia el modelo entero.

La definición del curso: **un cliente abandonó si lleva más de 180 días sin comprar respecto de la
última fecha del archivo.** Los 180 días no son sagrados; son la respuesta a una pregunta de negocio
—«¿a partir de cuándo damos por perdido a un cliente?»— y en tu caso puede ser 30, 90 o 365.

In [ ]:
ventas = pd.read_csv(DATOS / "ventas_limpias.csv", parse_dates=["fecha"])
clientes = pd.read_csv(DATOS / "clientes.csv", parse_dates=["fecha_alta"])
productos = pd.read_csv(DATOS / "productos.csv")
clientes["ciudad"] = clientes["ciudad"].str.strip().str.title().replace({"Guayaquíl": "Guayaquil"})

v = ventas.merge(productos[["producto_id", "costo_unitario"]], on="producto_id",
                 how="left", validate="m:1")
v["monto"] = v["cantidad"] * v["precio_unitario"] * (1 - v["descuento"])
v["margen"] = v["monto"] - v["cantidad"] * v["costo_unitario"]
compras = v[~v["es_devolucion"]]

HOY = v["fecha"].max()
VENTANA = 180
ultima_compra = compras.groupby("cliente_id")["fecha"].max()
dias_sin_comprar = (HOY - ultima_compra).dt.days

print(f"última fecha del archivo        : {HOY:%d-%m-%Y}")
print(f"última compra registrada        : {compras['fecha'].max():%d-%m-%Y} "
      "(las de julio son notas de crédito)")
print(f"clientes con al menos una compra: {len(ultima_compra):,} de {len(clientes):,} del padrón\n")
print(f"abandono (> {VENTANA} días sin comprar): {(dias_sin_comprar > VENTANA).mean():.2%} "
      f"= {(dias_sin_comprar > VENTANA).sum():,} clientes")
print(f"se quedan                             : {(dias_sin_comprar <= VENTANA).mean():.2%}\n")
print("Cómo cambia la cifra si cambias la definición:")
for w in [90, 120, 180, 270, 365]:
    print(f"  > {w:3d} días → {(dias_sin_comprar > w).mean():6.2%} de abandono")

⚠️ **La primera trampa está aquí y es de fuga de información.** Si defines el abandono con los últimos
180 días y después calculas las variables predictoras con **todo** el historial —que incluye esos mismos
180 días— el modelo ya sabe la respuesta: un cliente que abandonó no compró nada en la ventana final,
así que su «frecuencia total» baja por construcción. El modelo saldría espectacular y en producción
fallaría, porque cuando quieras usarlo aún no habrán pasado esos 180 días.

La solución es partir el tiempo en dos y no cruzar la línea:

```
|--------------- ventana de observación ---------------|---- ventana de resultado ----|
   se calculan las variables predictoras (hasta el corte)   se observa si compró o no
2024-01-01                                        2026-01-19                   2026-07-18
```

**Todo lo que entra al modelo se calcula con datos anteriores al corte. La etiqueta se mira después.**
Esta es la disciplina que separa un modelo que funciona de uno que solo lo parece.

In [ ]:
CORTE = HOY - pd.Timedelta(days=VENTANA)
historia = compras[compras["fecha"] <= CORTE]
g = historia.groupby("cliente_id")

X = pd.DataFrame({
    "frecuencia": g["factura_id"].nunique(),
    "monto": g["monto"].sum(),
    "primera": g["fecha"].min(),
    "ultima": g["fecha"].max(),
})
X["margen"] = v[v["fecha"] <= CORTE].groupby("cliente_id")["margen"].sum().reindex(X.index)
X["recencia_corte"] = (CORTE - X["ultima"]).dt.days
X["antiguedad"] = (CORTE - X["primera"]).dt.days
X["ticket_medio"] = X["monto"] / X["frecuencia"]
X = X.join(clientes.set_index("cliente_id")[["ciudad", "tipo_cliente", "canal_captacion"]])
X["abandono"] = (dias_sin_comprar > VENTANA).reindex(X.index).astype(int)

# Valor anual del cliente: margen de los doce meses anteriores al corte. Se usa solo para el dinero.
ultimo_ano = v[(v["fecha"] > CORTE - pd.Timedelta(days=365)) & (v["fecha"] <= CORTE)]
X["margen_anual"] = ultimo_ano.groupby("cliente_id")["margen"].sum().reindex(X.index).fillna(0)

NUMERICAS = ["recencia_corte", "frecuencia", "monto", "margen", "antiguedad", "ticket_medio"]
CATEGORICAS = ["ciudad", "tipo_cliente", "canal_captacion"]

print(f"corte de observación: {CORTE:%d-%m-%Y}")
print(f"clientes modelables (con al menos una compra antes del corte): {len(X):,}")
print(f"tasa de abandono en el conjunto modelable: {X['abandono'].mean():.2%}\n")
print(X.groupby("abandono")[NUMERICAS].mean().T
      .rename(columns={0: "se queda", 1: "abandona"})
      .assign(**{"razón": lambda d: d["abandona"] / d["se queda"]})
      .to_string(float_format=lambda x: f"{x:,.2f}"))

In [ ]:
# ✅ Comprobación 1 · la ventana de observación quedó bien cortada
assert len(X) == 1687, f"Deberías tener 1 687 clientes modelables y tienes {len(X)}"
assert abs(X["abandono"].mean() - 0.3266) < 0.005, \
    "La tasa de abandono no coincide con el 32,66 % del curso: revisa VENTANA y la fecha de corte"
assert X["recencia_corte"].min() >= 0, \
    "Hay recencias negativas: alguna compra posterior al corte se coló en la ventana de observación"
assert historia["fecha"].max() <= CORTE, \
    "La ventana de observación incluye datos posteriores al corte: eso es fuga de información"

print("Comprobación 1 superada ✓  1 687 clientes, 32,66 % de abandono, sin fuga entre ventanas")

📌 **1 687 clientes modelables y un 32,66 % de abandono.** Por cada cliente que se va hay dos que se
quedan: no es el 2 % de un problema de fraude, pero ya basta para que la exactitud empiece a mentir.

La tabla de medias adelanta lo que el modelo va a encontrar. El que abandona, **medido antes del
corte**, ya tenía la mitad de facturas (4,80 contra 8,91), un tercio del monto acumulado (513,60 contra
1 760,88) y llevaba **más del doble de días sin comprar** (187,06 contra 82,28). El abandono no es un
rayo: es un proceso que deja huella, y esa huella es lo que el modelo aprende. Fíjate en `antiguedad`
—483,60 contra 527,87 días—: casi no distingue, y eso también es un resultado.

## 2. Probabilidad en lugar de etiqueta

Una regresión lineal aplicada a un sí/no daría predicciones de 1,4 o de −0,2, que no significan nada.
La **regresión logística** resuelve el problema pasando la recta por una función que aplasta cualquier
número al intervalo de 0 a 1:

> probabilidad = 1 / (1 + e^(−(b₀ + b₁·x₁ + b₂·x₂ + …)))

Lo que el modelo devuelve es **una probabilidad por cliente**, no una etiqueta. La etiqueta aparece
después, cuando alguien elige un umbral, y esa elección es una decisión de negocio —sección 6—, no un
detalle técnico.

Los coeficientes se leen en **razón de momios** (`exp(coef)`): cuánto se multiplican las probabilidades
relativas de abandonar cuando la variable sube una desviación estándar.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

prep_lineal = ColumnTransformer([
    ("num", StandardScaler(), NUMERICAS),
    ("cat", OneHotEncoder(drop="first"), CATEGORICAS)])
# El árbol no necesita escalado: sin él, los cortes se leen en días y en dólares.
prep_arbol = ColumnTransformer([
    ("num", "passthrough", NUMERICAS),
    ("cat", OneHotEncoder(drop="first"), CATEGORICAS)])

X_ent, X_pru, y_ent, y_pru = train_test_split(
    X[NUMERICAS + CATEGORICAS], X["abandono"], test_size=0.30,
    stratify=X["abandono"], random_state=SEED)

logistica = Pipeline([("prep", prep_lineal),
                      ("clf", LogisticRegression(max_iter=3000, random_state=SEED))]).fit(X_ent, y_ent)

nombres = [n.split("__")[1] for n in logistica.named_steps["prep"].get_feature_names_out()]
coefs = pd.DataFrame({"variable": nombres, "coeficiente": logistica.named_steps["clf"].coef_[0]})
coefs["razón de momios"] = np.exp(coefs["coeficiente"])
coefs["efecto"] = np.where(coefs["coeficiente"] > 0, "↑ abandona más", "↓ abandona menos")

print(f"entrenamiento {len(X_ent):,} clientes · prueba {len(X_pru):,} clientes "
      f"(abandono {y_ent.mean():.2%} y {y_pru.mean():.2%})\n")
print(coefs.reindex(coefs["coeficiente"].abs().sort_values(ascending=False).index)
      .to_string(index=False, float_format=lambda v: f"{v:+.4f}"))
print(f"\nintercepto: {logistica.named_steps['clf'].intercept_[0]:+.4f}")

prob_ent = logistica.predict_proba(X_ent)[:, 1]
prob_pru = logistica.predict_proba(X_pru)[:, 1]

print(f"probabilidad media que asigna el modelo a los que SÍ abandonaron : "
      f"{prob_pru[y_pru.values == 1].mean():.4f}")
print(f"probabilidad media que asigna a los que se quedaron              : "
      f"{prob_pru[y_pru.values == 0].mean():.4f}")
print(f"clientes de prueba con probabilidad mayor que 0,5                : "
      f"{(prob_pru >= 0.5).sum()} de {len(prob_pru)}")
print(f"solapamiento: {((prob_pru > 0.3) & (prob_pru < 0.6)).mean():.1%} de la cartera cae en la "
      "zona donde las dos clases se mezclan")

📌 **La variable que más pesa es la frecuencia, con una razón de momios de 0,4744.** Cada desviación
estándar más de facturas —unas 6,7— multiplica por 0,47 las probabilidades relativas de abandonar: **las
parte casi por la mitad**. Después vienen ser minorista (1,9208) y la recencia al corte (1,7801).

Mira las dos probabilidades medias, porque son la lección de la sección: el modelo separa —0,4540 para
los que abandonan contra 0,2740 para los que se quedan— pero **las dos distribuciones se solapan** y no
hay ningún punto donde cortar limpio. **La pregunta no es cómo evitarlo, sino cuál de los dos errores
prefieres cometer**, y eso solo se contesta con dinero.

## 3. Dos modelos: el que da probabilidades y el que se lee en voz alta

Un **árbol de decisión** parte la cartera con preguntas encadenadas, eligiendo cada pregunta para que
los grupos resultantes sean lo más puros posible. Su ventaja no es la precisión: es que **el resultado
se lee como un manual de procedimiento** y se discute con el equipo comercial sin traducir nada. La
profundidad es la palanca: sin límite memoriza; con profundidad 3 caben ocho reglas en una pantalla. El
árbol dibujado y traducido a instrucciones de vendedor está en el apéndice; aquí entra directo a la
comparación, que es lo que decide.

## 4. Contra la línea base, con validación cruzada estratificada

La regla de la semana 11 no cambia porque el problema sea de clasificación: **primero la línea base**,
que aquí es `DummyClassifier` y tiene dos versiones que hay que ver juntas.

Con clases desbalanceadas se usa `StratifiedKFold`, que mantiene la proporción de abandono en cada
pliegue. Con `KFold` normal puede tocarte un pliegue con el 45 % de abandono y otro con el 20 %, y la
comparación deja de significar nada.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score

# zero_division=0: la línea base nunca predice "abandona" y sin esto sklearn avisa en cada pliegue.
METRICAS = {"accuracy": "accuracy", "roc_auc": "roc_auc",
            "precision": make_scorer(precision_score, zero_division=0),
            "recall": make_scorer(recall_score, zero_division=0),
            "f1": make_scorer(f1_score, zero_division=0)}

candidatos = {
    "línea base · 'nadie abandona'": DummyClassifier(strategy="most_frequent"),
    "línea base · al azar con la misma proporción": DummyClassifier(strategy="stratified",
                                                                   random_state=SEED),
    "regresión logística": Pipeline([("prep", prep_lineal),
                                     ("clf", LogisticRegression(max_iter=3000, random_state=SEED))]),
    "árbol · profundidad 3": Pipeline([("prep", prep_arbol),
                                       ("clf", DecisionTreeClassifier(max_depth=3,
                                                                      random_state=SEED))]),
    "árbol · sin podar": Pipeline([("prep", prep_arbol),
                                   ("clf", DecisionTreeClassifier(random_state=SEED))]),
}

filas = {}
for nombre, modelo in candidatos.items():
    cv = cross_validate(modelo, X[NUMERICAS + CATEGORICAS], X["abandono"],
                        cv=skf, scoring=METRICAS)
    filas[nombre] = {m: cv[f"test_{m}"].mean() for m in METRICAS}

tabla = pd.DataFrame(filas).T[["accuracy", "precision", "recall", "f1", "roc_auc"]].rename(columns={
    "accuracy": "exactitud", "precision": "precisión", "recall": "sensibilidad",
    "f1": "F1", "roc_auc": "área bajo la curva ROC"})
print(f"validación cruzada estratificada de 5 pliegues · {len(X):,} clientes · "
      f"abandono {X['abandono'].mean():.2%}\n")
print(tabla.to_string(float_format=lambda v: f"{v:.4f}"))

📌 **La línea base que dice que nadie abandona acierta el 67,34 % de las veces**, sin mirar un solo
dato del cliente. La regresión logística, el mejor modelo de la tabla, acierta el 73,20 %: **menos de
seis puntos de mejora**. Con esa cifra el informe concluiría que el modelo no vale la pena, y sería
exactamente la conclusión equivocada. Las otras columnas lo demuestran:

- La línea base tiene **sensibilidad 0,0000**: no detecta ni un cliente que se va. Su 67 % de exactitud
  solo mide que el 67 % de la cartera se queda.
- El área bajo la curva ROC de la línea base es **0,5000** —azar puro— y la de la logística **0,7581**.
  Esa métrica sí distingue, porque mide la capacidad de **ordenar**.
- El **árbol sin podar** saca 0,6497 de exactitud, **por debajo de la línea base**. Memorizó el
  entrenamiento y no generalizó nada.

**El modelo que se lleva a producción es la regresión logística** (0,7581 contra 0,7295 del árbol
podado), y el árbol se queda como material de comunicación con el equipo comercial.

## 5. La matriz de confusión traducida a dinero

Las cuatro celdas de la matriz cuestan cosas distintas, y hasta que no se les pone precio no hay forma
de elegir métrica ni umbral. Los precios salen de la contabilidad y de una conversación con el gerente
comercial, **nunca del modelo**, y se escriben antes de mirar los resultados.

| celda | qué pasa en la realidad | precio |
|---|---|---|
| **Verdadero negativo** · dijo que se queda y se queda | No se le llama, sigue comprando | 0 |
| **Falso positivo** · dijo que se va y se queda | Llamada gastada en quien no la necesitaba | −costo de contacto |
| **Falso negativo** · dijo que se queda y se va | Se pierde el cliente sin intentar nada | −valor anual del cliente |
| **Verdadero positivo** · dijo que se va y se va | Se le llama; se recupera con probabilidad *p* | −costo de contacto − (1−*p*)·valor |

Los dos supuestos que hay que declarar: **cuánto cuesta un contacto** y **qué proporción de los
contactados se retiene**. Si el grupo no los puede defender, el análisis no se sostiene.

In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score, accuracy_score

# --- Supuestos declarados ---
COSTO_CONTACTO = 12.00   # llamada del vendedor + incentivo pequeño, en dólares
TASA_RETENCION = 0.30    # de los que iban a irse y reciben la llamada, se queda el 30 %

# El valor del cliente se estima SOLO con el entrenamiento, como cualquier otro parámetro.
VALOR_CLIENTE = X.loc[X_ent.index, "margen_anual"][y_ent == 1].mean()

print(f"costo de contactar a un cliente      : {COSTO_CONTACTO:>8,.2f}")
print(f"tasa de retención de la llamada      : {TASA_RETENCION:>8.0%}")
print(f"valor anual de un cliente que se va  : {VALOR_CLIENTE:>8,.2f}  "
      "(margen de sus últimos 12 meses)")
print(f"ahorro esperado por llamada acertada : "
      f"{TASA_RETENCION * VALOR_CLIENTE:>8,.2f}")
print(f"\n→ una llamada se paga si la probabilidad de acertar supera "
      f"{COSTO_CONTACTO / (TASA_RETENCION * VALOR_CLIENTE):.2%}")
print(f"→ la tasa de abandono de la cartera es {X['abandono'].mean():.2%}: "
      "llamar al azar NO se paga")


def valor_de_la_matriz(y_real, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_real, y_pred, labels=[0, 1]).ravel()
    return pd.Series({
        "VN · acertó que se queda": tn * 0.0,
        "FP · llamada desperdiciada": -COSTO_CONTACTO * fp,
        "FN · se fue sin que lo llamaran": -VALOR_CLIENTE * fn,
        "VP · llamada acertada": -COSTO_CONTACTO * tp - (1 - TASA_RETENCION) * VALOR_CLIENTE * tp,
    })

y_real = y_pru.values
pred_05 = (prob_pru >= 0.50).astype(int)
tn, fp, fn, tp = confusion_matrix(y_real, pred_05, labels=[0, 1]).ravel()

matriz = pd.DataFrame([[tn, fp], [fn, tp]],
                      index=["se queda de verdad", "abandona de verdad"],
                      columns=["el modelo dice 'se queda'", "el modelo dice 'abandona'"])
print(f"Matriz de confusión en clientes · conjunto de prueba, umbral 0,50 "
      f"({len(y_real)} clientes)\n")
print(matriz.to_string())

dinero = pd.DataFrame([[0.0, -COSTO_CONTACTO * fp],
                       [-VALOR_CLIENTE * fn,
                        -COSTO_CONTACTO * tp - (1 - TASA_RETENCION) * VALOR_CLIENTE * tp]],
                      index=matriz.index, columns=matriz.columns)
print(f"\nLa misma matriz en dólares\n")
print(dinero.to_string(float_format=lambda v: f"{v:,.2f}"))

politicas = pd.DataFrame({
    "no hacer nada": valor_de_la_matriz(y_real, np.zeros_like(y_real)),
    "llamar a toda la cartera": valor_de_la_matriz(y_real, np.ones_like(y_real)),
    "llamar a quien señala el modelo (umbral 0,50)": valor_de_la_matriz(y_real, pred_05),
}).T
politicas["TOTAL"] = politicas.sum(axis=1)
politicas["retorno frente a no hacer nada"] = (politicas["TOTAL"]
                                               - politicas.loc["no hacer nada", "TOTAL"])
print(f"\nLas tres políticas posibles, en dólares sobre los {len(y_real)} clientes de prueba\n")
print(politicas.to_string(float_format=lambda v: f"{v:,.2f}"))

📌 **Llamar a toda la cartera pierde 883,65 dólares frente a no hacer nada. Llamar a quien señala el
modelo gana 739,64.** Mismo equipo, mismo guion, mismo presupuesto por llamada: lo único que cambia es
a quién se llama. La aritmética cabe en tres líneas:

- Cada llamada cuesta 12,00 y ahorra 31,33 **si acierta** (el 30 % de 104,42), así que se paga si la
  probabilidad de acertar supera **38,31 %**.
- La tasa de abandono de la cartera es **32,66 %**, por debajo de ese punto de equilibrio: por eso la
  campaña masiva destruye valor.
- El modelo, con umbral 0,50, acierta el 63,16 % de las veces que llama.

**El modelo no vale por acertar más: vale por concentrar los aciertos donde hay presupuesto.** Y fíjate
en la celda que más dinero mueve, los falsos negativos: 11 069 dólares de clientes que se fueron sin que
nadie los llamara. Bajar esa celda es lo que hace el umbral.

## 6. El umbral es una palanca gerencial

El 0,50 no tiene nada de especial: es el corte que `scikit-learn` usa por defecto y casi nunca es el
correcto. Bajarlo significa llamar a más gente —más aciertos y más llamadas desperdiciadas— y subirlo,
lo contrario. Como ya tenemos los precios, no hace falta discutir: se prueba el rango entero y gana el
dinero.

In [ ]:
def evaluar_umbral(umbral, prob=prob_pru, real=y_real):
    pred = (prob >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(real, pred, labels=[0, 1]).ravel()
    total = (-COSTO_CONTACTO * (tp + fp) - (1 - TASA_RETENCION) * VALOR_CLIENTE * tp
             - VALOR_CLIENTE * fn)
    return {"umbral": umbral, "llamadas": tp + fp, "% de la cartera": (tp + fp) / len(real),
            "aciertos (VP)": tp, "desperdiciadas (FP)": fp, "se escapan (FN)": fn,
            "precisión": tp / (tp + fp) if tp + fp else np.nan,
            "sensibilidad": tp / (tp + fn),
            "retorno": total - (-VALOR_CLIENTE * real.sum())}


barrido = pd.DataFrame([evaluar_umbral(u) for u in np.arange(0.05, 0.96, 0.05)])
mejor = barrido.loc[barrido["retorno"].idxmax()]

print(barrido.to_string(index=False, float_format=lambda v: f"{v:,.4f}"))
print(f"\numbral que maximiza el retorno : {mejor['umbral']:.2f} → "
      f"{mejor['retorno']:,.2f} dólares con {int(mejor['llamadas'])} llamadas")
print(f"umbral por defecto de sklearn  : 0.50 → "
      f"{barrido.loc[barrido['umbral'].round(2) == 0.50, 'retorno'].iloc[0]:,.2f} dólares")
print(f"diferencia por mover un número : "
      f"{mejor['retorno'] - barrido.loc[barrido['umbral'].round(2) == 0.50, 'retorno'].iloc[0]:,.2f}")

In [ ]:
# ✅ Comprobación 2 · el umbral que maximiza el dinero no es el de fábrica
retorno_05 = barrido.loc[barrido["umbral"].round(2) == 0.50, "retorno"].iloc[0]

assert abs(VALOR_CLIENTE - 104.42) < 1.0, \
    "El valor anual del cliente no coincide: revisa que margen_anual se calcule solo antes del corte"
assert abs(mejor["umbral"] - 0.40) < 0.001, \
    f"El umbral óptimo debería ser 0,40 y te salió {mejor['umbral']:.2f}: revisa los supuestos de dinero"
assert mejor["retorno"] > retorno_05 > 0, \
    "El umbral óptimo tiene que rendir más que el 0,50 de fábrica, y los dos más que no hacer nada"
assert barrido["retorno"].iloc[0] < 0, \
    "Con el umbral más bajo se llama a media cartera y el retorno tiene que ser negativo"

print(f"Comprobación 2 superada ✓  óptimo {mejor['umbral']:.2f} → {mejor['retorno']:,.2f} "
      f"frente a {retorno_05:,.2f} del 0,50 de fábrica")

📌 **El umbral óptimo es 0,40 y devuelve 1 017,45 dólares; el 0,50 de fábrica devuelve 739,64.** Mover
un número que nadie discute en las reuniones vale 277,80 dólares sobre 507 clientes —un 37,6 % más de
retorno— y la diferencia se escala con la cartera.

Con umbrales por debajo de 0,15 el modelo llama a tanta gente que la precisión cae por debajo del
38,31 % de equilibrio y **el retorno se vuelve negativo**. Y entre 0,30 y 0,45 el retorno se mueve entre
830 y 1 017: **no hay que afinar el umbral al segundo decimal**, hay que no dejarlo en 0,50 por inercia.
Las dos curvas del barrido están en el apéndice.

## 7. La curva de ganancia: el presupuesto manda

En la vida real el umbral no lo elige el modelo, lo elige el presupuesto. El gerente dice *«tengo equipo
para llamar a 100 clientes este mes»* y la pregunta pasa a ser: **de los que se van, ¿a cuántos alcanzo
si llamo a los 100 con la puntuación más alta?** Eso es la curva de ganancia: ordenar la cartera por
probabilidad de mayor a menor y acumular cuántos abandonos vas capturando.

In [ ]:
orden = np.argsort(-prob_pru)
real_ordenado = y_real[orden]
n = len(real_ordenado)

ganancia = pd.DataFrame({
    "% de la cartera contactada": (np.arange(n) + 1) / n,
    "% de los abandonos capturado": np.cumsum(real_ordenado) / real_ordenado.sum(),
})
ganancia["lift"] = (ganancia["% de los abandonos capturado"]
                    / ganancia["% de la cartera contactada"])

hitos = []
for q in [0.05, 0.10, 0.20, 0.30, 0.50, 1.00]:
    k = int(round(q * n))
    capturados = int(real_ordenado[:k].sum())
    retorno = (-COSTO_CONTACTO * k - (1 - TASA_RETENCION) * VALOR_CLIENTE * capturados
               - VALOR_CLIENTE * (real_ordenado.sum() - capturados)
               + VALOR_CLIENTE * real_ordenado.sum())
    hitos.append({"% cartera": q, "clientes llamados": k, "abandonos capturados": capturados,
                  "% de abandonos": capturados / real_ordenado.sum(),
                  "lift": (capturados / real_ordenado.sum()) / q, "retorno": retorno})
hitos = pd.DataFrame(hitos)
print(hitos.to_string(index=False, float_format=lambda v: f"{v:,.4f}"))

BUDGET = 0.20
k20 = int(round(BUDGET * n))
print(f"\nCon presupuesto para el {BUDGET:.0%} de la cartera ({k20} llamadas de {n}):")
print(f"  · se captura el {hitos.loc[hitos['% cartera'] == BUDGET, '% de abandonos'].iloc[0]:.2%} "
      f"de los abandonos ({int(hitos.loc[hitos['% cartera'] == BUDGET, 'abandonos capturados'].iloc[0])} "
      f"de {int(real_ordenado.sum())})")
print(f"  · eso es {hitos.loc[hitos['% cartera'] == BUDGET, 'lift'].iloc[0]:.2f} veces más "
      "que llamar a 101 clientes al azar")
print(f"  · retorno esperado {hitos.loc[hitos['% cartera'] == BUDGET, 'retorno'].iloc[0]:,.2f} dólares")
print(f"  · probabilidad del último cliente de la lista: {np.sort(prob_pru)[::-1][k20 - 1]:.4f} "
      "← este es el umbral que impone el presupuesto")

fig, ax = plt.subplots(figsize=(10, 4.4))
ax.plot(ganancia["% de la cartera contactada"] * 100,
        ganancia["% de los abandonos capturado"] * 100,
        color="#4C72B0", linewidth=2.5, label="modelo")
ax.plot([0, 100], [0, 100], color="grey", linestyle="--", linewidth=1.5,
        label="llamar al azar")
ax.plot([0, X["abandono"].mean() * 100, 100], [0, 100, 100], color="#55A868",
        linestyle=":", linewidth=1.5, label="modelo perfecto")
pct20 = hitos.loc[hitos["% cartera"] == 0.20, "% de abandonos"].iloc[0] * 100
ax.axvline(20, color="#C44E52", linestyle="--", linewidth=1.5)
ax.annotate(f"con el 20 % de la cartera\nse captura el {pct20:.1f} % de los abandonos",
            xy=(20, pct20), xytext=(30, 28), fontsize=10,
            arrowprops=dict(arrowstyle="->", color="#C44E52"))
ax.set_xlabel("% de la cartera contactada, ordenada por probabilidad")
ax.set_ylabel("% de los abandonos capturados")
ax.set_title("Llamando al 20 % de la cartera se alcanza al doble de fugas que llamando al azar",
             fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

📌 **Con el 20 % de la cartera se captura el 38,55 % de los abandonos: un lift de 1,93.** Dicho para el
comité: *«con presupuesto para llamar a 101 de los 507 clientes, el modelo alcanza a 64 de los 166 que
se iban a ir; llamando al azar alcanzaríamos a 33. El retorno esperado es de 792,95 dólares en el
conjunto de prueba»*.

Y hay un número más que vale oro: **la probabilidad del cliente número 101 es 0,4823.** Ese es el
umbral, y no lo eligió el analista: lo impuso el presupuesto. Así se trabaja en la práctica —**primero
cuántas llamadas hay, después el corte**—, y por eso el umbral es una decisión gerencial disfrazada de
parámetro. Fíjate también en la forma de la curva: el 10 % mejor de la cartera tiene un lift de 2,23 y
el 50 %, de 1,53. **El valor del modelo está concentrado en la cabeza de la lista.**

## 8. El fragmento que te dio la IA

Le pediste a un asistente que evaluara el modelo de abandono. Devolvió esto, con su comentario
entusiasta incluido. El código corre, no da ningún error y **el resultado es imposible**.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
#  Código generado por un asistente de IA. Corre sin errores.
#  «¡Excelente! El modelo alcanza un rendimiento sobresaliente en todas
#   las métricas. Está listo para producción.»
# ─────────────────────────────────────────────────────────────────────────
X_ia = X[NUMERICAS + CATEGORICAS]
y_ia = X["abandono"]

modelo_ia = Pipeline([
    ("prep", ColumnTransformer([("num", "passthrough", NUMERICAS),
                                ("cat", OneHotEncoder(drop="first"), CATEGORICAS)])),
    ("clf", DecisionTreeClassifier(random_state=42)),
])
modelo_ia.fit(X_ia, y_ia)
pred_ia = modelo_ia.predict(X_ia)

print("=== Resultados del modelo de abandono ===")
print(f"Exactitud    : {accuracy_score(y_ia, pred_ia):.4f}")
print(f"Precisión    : {precision_score(y_ia, pred_ia):.4f}")
print(f"Sensibilidad : {recall_score(y_ia, pred_ia):.4f}")
print(f"F1           : {f1_score(y_ia, pred_ia):.4f}")
print(f"Área ROC     : {roc_auc_score(y_ia, modelo_ia.predict_proba(X_ia)[:, 1]):.4f}")
print("\nEl modelo identifica correctamente a todos los clientes en riesgo.")

### 🌶️ Ejercicio 1 — Guiado

Ese fragmento tiene **un** error, está en dos líneas, y no es de sintaxis. Tu trabajo:

1. **Encuéntralo.** Escribe en una celda de texto qué línea lo produce y por qué el resultado que
   imprime es imposible en un problema real.
2. **Corrígelo** de las dos formas válidas: (a) con `train_test_split` apartando un conjunto de prueba
   y (b) con `cross_validate` y `StratifiedKFold`, que es la buena porque usa todos los datos.
3. **Cuantifica el engaño.** Construye una tabla con tres columnas —lo que reportó la IA, lo que da la
   evaluación honesta y la diferencia— para exactitud, sensibilidad y área bajo la curva ROC.
4. **Compara la cifra honesta con la línea base** de la sección 4. Contesta en una frase si ese modelo
   se puede poner en producción.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: mira qué conjunto de datos entra en .fit() y cuál entra en .predict(). ¿Son el mismo?
# Pista 2: un árbol sin max_depth crece hasta que cada hoja tiene un solo cliente. Sobre los datos
#          que ya vio no puede fallar. Sobre datos nuevos no tiene nada aprendido
# Pista 3: cross_validate(modelo_ia, X_ia, y_ia, cv=skf, scoring=["accuracy","recall","roc_auc"])
# Pista 4: la exactitud honesta que te va a salir es MENOR que la de la línea base de la sección 4.
#          Ese es el titular del ejercicio

### 🔥 Desafío · dividido en dos

**El valor del cliente no es el mismo para todos, y eso cambia a quién llamas.** Hasta aquí usamos un
valor único de 104,42 dólares para cualquier cliente que abandona. La columna `margen_anual` tiene el
valor real de cada uno.

**Parte A · en clase (20 min).** Rehaz el retorno de las **tres políticas** de la sección 5 y el
**umbral óptimo** usando `margen_anual` cliente a cliente en lugar del promedio. Dos preguntas a
contestar con el número delante: ¿cambia el umbral que maximiza el dinero?, ¿cambia la política que
gana?

**Parte B · para casa (35 min).** La curva de ganancia **en dinero** en lugar de en número de clientes,
las dos curvas superpuestas en un solo gráfico, y la pregunta incómoda: si el modelo ordena bien a los
clientes que se van pero **los que se van son los que menos valen**, ¿la campaña sigue pagando? Si la
respuesta es que no, esa es tu conclusión y se escribe tal cual en la entrega.

In [ ]:
# TU CÓDIGO AQUÍ
# --- Parte A (en clase) ---
# Pista 1: valores = X.loc[X_pru.index, "margen_anual"].values  ← alineado con prob_pru
# Pista 2: en evaluar_umbral, sustituye VALOR_CLIENTE * fn por valores[(pred == 0) & (real == 1)].sum()
#          y haz lo mismo con los verdaderos positivos
#
# --- Parte B (para casa) ---
# Pista 3: la curva de ganancia en dinero es np.cumsum(np.where(real_ordenado==1, valores_ordenados, 0))
#          dividido por el total del dinero en riesgo
# Pista 4: compara el % de abandonos capturado con el % de dinero capturado en el top 20 %.
#          Si el segundo es mucho más bajo, tienes un hallazgo y hay que reportarlo

### 🎯 Reto en clase (15 min)

En equipos y contra reloj: **la subasta del presupuesto**. El docente anuncia un presupuesto en
llamadas —50, 100 o 200— y cada equipo tiene siete minutos para entregar tres cosas en una sola
diapositiva: el umbral que corresponde a ese presupuesto, el número de abandonos que espera capturar y
el retorno esperado en dólares. A los siete minutos, el docente **cambia el costo de contacto a 40
dólares** y hay tres minutos para rehacerlo. Gana el equipo cuyo retorno siga siendo positivo con el
costo nuevo y pueda explicar en una frase qué cambió y por qué.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: el umbral que impone un presupuesto de k llamadas es np.sort(prob_pru)[::-1][k-1]
# Pista 2: con COSTO_CONTACTO = 40 el punto de equilibrio pasa de 38,31 % a
#          40 / (0.30 * VALOR_CLIENTE). Calcúlalo antes de tocar el umbral
# Pista 3: si con el costo nuevo NINGÚN umbral da retorno positivo, la respuesta correcta es
#          "no se hace la campaña". Es una respuesta válida y vale el reto

## La trampa de hoy

⚠️ **Optimizar la exactitud en un problema desbalanceado.** Es la trampa que hace que se aprueben
modelos inútiles y que se descarten modelos que valen dinero, y las dos cosas en la misma reunión.

Tres modelos, tres exactitudes, y la única columna que importa al final.

In [ ]:
comparacion = pd.DataFrame({
    "exactitud (validación cruzada)": [tabla.loc["línea base · 'nadie abandona'", "exactitud"],
                                       tabla.loc["árbol · sin podar", "exactitud"],
                                       tabla.loc["regresión logística", "exactitud"]],
    "sensibilidad": [tabla.loc["línea base · 'nadie abandona'", "sensibilidad"],
                     tabla.loc["árbol · sin podar", "sensibilidad"],
                     tabla.loc["regresión logística", "sensibilidad"]],
    "área bajo la curva ROC": [tabla.loc["línea base · 'nadie abandona'", "área bajo la curva ROC"],
                               tabla.loc["árbol · sin podar", "área bajo la curva ROC"],
                               tabla.loc["regresión logística", "área bajo la curva ROC"]],
}, index=["línea base 'nadie abandona'", "árbol sin podar", "regresión logística"])

# Retorno de cada uno sobre el conjunto de prueba, con los precios de la sección 5
arbol_sucio = Pipeline([("prep", prep_arbol),
                        ("clf", DecisionTreeClassifier(random_state=SEED))]).fit(X_ent, y_ent)
retornos = []
for pred in [np.zeros_like(y_real), arbol_sucio.predict(X_pru),
             (prob_pru >= mejor["umbral"]).astype(int)]:
    tn_, fp_, fn_, tp_ = confusion_matrix(y_real, pred, labels=[0, 1]).ravel()
    retornos.append(-COSTO_CONTACTO * (tp_ + fp_) - (1 - TASA_RETENCION) * VALOR_CLIENTE * tp_
                    - VALOR_CLIENTE * fn_ + VALOR_CLIENTE * y_real.sum())
comparacion["retorno en dólares"] = retornos

BASE, ARBOL = comparacion.index[0], comparacion.index[1]
comparacion["puesto por exactitud"] = (comparacion["exactitud (validación cruzada)"]
                                       .rank(ascending=False).astype(int))
comparacion["puesto por dinero"] = comparacion["retorno en dólares"].rank(ascending=False).astype(int)
print(comparacion.to_string(float_format=lambda v: f"{v:,.4f}"))
print(f"\nPor exactitud : la {BASE} va en el puesto "
      f"{comparacion.loc[BASE, 'puesto por exactitud']} y el {ARBOL} en el "
      f"{comparacion.loc[ARBOL, 'puesto por exactitud']}")
print(f"Por dinero    : al revés — puesto {comparacion.loc[BASE, 'puesto por dinero']} "
      f"y puesto {comparacion.loc[ARBOL, 'puesto por dinero']}")
print(f"\nY el fragmento de la sección 8 reportaba exactitud 1,0000 para el árbol sin podar,")
print(f"cuyo desempeño honesto es {comparacion.loc['árbol sin podar', 'exactitud (validación cruzada)']:.4f}: "
      f"{1 - comparacion.loc['árbol sin podar', 'exactitud (validación cruzada)']:.4f} de exageración,")
print("y por debajo de la línea base que no mira ningún dato.")

📌 **La exactitud pone la línea base por delante del árbol; el dinero los pone al revés.** La línea base
saca 0,6734 sin mirar nada y no genera un solo dólar. El árbol sin podar saca 0,6497 —peor— y aun así
genera dinero, porque aunque se equivoque mucho se equivoca **en la dirección útil**: llama a gente. Y
la logística con el umbral en 0,40 gana en las tres columnas que importan.

**La exactitud premia a la clase mayoritaria.** Con el 67 % de la cartera quedándose, decir «se queda»
da un 67 % de aciertos que no contiene ninguna información; en un problema al 2 % el mismo truco da un
98 %. Qué mirar en su lugar, en orden: **la línea base**, **el área bajo la curva ROC** para comparar
modelos, **la sensibilidad y la precisión juntas** —son las que se convierten en llamadas— y **el
dinero**, la única que contesta si se hace la campaña. La regla que resume el laboratorio: **la
exactitud no se reporta sola jamás.**

## Entregable

Sube `lab_14_apellido.ipynb` con:

- La variable de abandono construida con **la ventana de observación separada de la de resultado**, con
  la fecha de corte declarada (19-01-2026) y la tasa resultante (32,66 % sobre 1 687 clientes).
- La logística y el árbol comparados por validación cruzada estratificada **contra las dos líneas
  base**, con la tabla completa de las cinco métricas.
- La **matriz de confusión en dólares** con los dos supuestos escritos y defendibles —12,00 de contacto
  y 30 % de retención— y las tres políticas comparadas: no hacer nada, llamar a todos (−883,65) y llamar
  a quien señala el modelo (+739,64).
- El **barrido de umbral** con el óptimo (0,40 → 1 017,45) frente al 0,50 de fábrica (739,64), y por qué
  la campaña masiva pierde dinero: 32,66 % de tasa contra 38,31 % de punto de equilibrio.
- La **curva de ganancia** con el hito del 20 % de la cartera: 38,55 % de los abandonos capturados,
  lift 1,93, umbral impuesto 0,4823.
- El fragmento de la IA **corregido**, con la tabla de cuánto exageraba: 1,0000 de exactitud reportada
  contra 0,6497 honesta.
- La recomendación de focalización del caso en cinco líneas: a quiénes se llama, con qué umbral, cuántos
  son, qué se espera capturar y cuánto vale.

## Para tu equipo

- La definición de la variable objetivo se **escribe y se discute con la empresa antes de modelar**.
  «Abandono» no significa lo mismo en un distribuidor que en una suscripción, y el 32,66 % cambia por
  completo con la ventana que elijan.
- Pregunten los dos números del dinero: **cuánto cuesta contactar a un cliente y cuántos se retienen**.
  Con esos dos números el umbral se calcula solo; sin ellos, cualquier umbral es una opinión.
- Entreguen la recomendación como una **lista de clientes ordenada**, no como un modelo. Lo que el
  gerente puede usar el lunes es un archivo con nombres y una columna de prioridad.

## Apéndice · para profundizar fuera de clase

Lo que sigue **no compite por los noventa minutos de clase**: es opcional y está aquí para quien quiera
llegar más lejos, o para copiarlo al informe del proyecto. Se ejecuta después de haber corrido todas
las celdas anteriores.

### A1 · Cómo se separan las variables entre los dos grupos

Los histogramas de las tres variables que más pesan y la tasa de abandono por ciudad, tipo de cliente y canal de captación. Ninguna separa sola: por eso hace falta un modelo.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.6))
for ax, (col, etiqueta) in zip(axes, [("recencia_corte", "días sin comprar al corte"),
                                      ("frecuencia", "facturas antes del corte"),
                                      ("ticket_medio", "ticket medio (dólares)")]):
    for etq, color, nombre in [(0, "#4C72B0", "se queda"), (1, "#C44E52", "abandona")]:
        datos = X.loc[X["abandono"] == etq, col]
        datos = datos[datos <= datos.quantile(0.98)]
        ax.hist(datos, bins=35, alpha=0.6, color=color, label=nombre, density=True)
    ax.set_xlabel(etiqueta)
    ax.set_yticks([])
    ax.legend(fontsize=8)
axes[0].set_title("Se separan bien", fontsize=11)
axes[1].set_title("Se separan bien", fontsize=11)
axes[2].set_title("Se solapan mucho", fontsize=11)
fig.suptitle("Ninguna variable separa sola: por eso hace falta un modelo", fontsize=12, y=1.04)
plt.tight_layout()
plt.show()

por_grupo = pd.concat([
    X.groupby("ciudad")["abandono"].agg(clientes="size", tasa="mean"),
    X.groupby("tipo_cliente")["abandono"].agg(clientes="size", tasa="mean"),
    X.groupby("canal_captacion")["abandono"].agg(clientes="size", tasa="mean"),
])
print("Tasa de abandono por grupo:\n")
print(por_grupo.to_string(float_format=lambda v: f"{v:,.4f}"))

### A2 · La sigmoide y el solapamiento de las probabilidades

A la izquierda, la función que convierte cualquier número en una probabilidad. A la derecha, la razón por la que no existe un umbral obvio.

In [ ]:
prob_ent = logistica.predict_proba(X_ent)[:, 1]
prob_pru = logistica.predict_proba(X_pru)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
z = np.linspace(-6, 6, 300)
axes[0].plot(z, 1 / (1 + np.exp(-z)), color="#4C72B0", linewidth=2.5)
axes[0].axhline(0.5, color="#C44E52", linestyle="--", linewidth=1)
axes[0].set_xlabel("suma lineal b₀ + b₁x₁ + …  (log-momios)")
axes[0].set_ylabel("probabilidad")
axes[0].set_title("La sigmoide: cualquier número entra, sale algo entre 0 y 1", fontsize=11)

for etq, color, nombre in [(0, "#4C72B0", "se queda"), (1, "#C44E52", "abandona")]:
    axes[1].hist(prob_pru[y_pru.values == etq], bins=28, alpha=0.65, color=color,
                 label=nombre, density=True)
axes[1].axvline(0.5, color="grey", linestyle="--", linewidth=1.5)
axes[1].set_xlabel("probabilidad de abandono que asigna el modelo")
axes[1].set_yticks([])
axes[1].legend(fontsize=9)
axes[1].set_title("Se solapan: por eso no hay un umbral obvio", fontsize=11)
plt.tight_layout()
plt.show()

print(f"probabilidad media que asigna el modelo a los que SÍ abandonaron : "
      f"{prob_pru[y_pru.values == 1].mean():.4f}")
print(f"probabilidad media que asigna a los que se quedaron              : "
      f"{prob_pru[y_pru.values == 0].mean():.4f}")
print(f"clientes de prueba con probabilidad mayor que 0,5                : "
      f"{(prob_pru >= 0.5).sum()} de {len(prob_pru)}")

### A3 · El árbol dibujado y traducido a reglas

El árbol de profundidad 3 de la sección 3, primero como texto y como dibujo, y después traducido a instrucciones que un vendedor puede aplicar con la libreta en la mano.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

arbol = Pipeline([("prep", prep_arbol),
                  ("clf", DecisionTreeClassifier(max_depth=3, random_state=SEED))]).fit(X_ent, y_ent)
nombres_arbol = [n.split("__")[1] for n in arbol.named_steps["prep"].get_feature_names_out()]

print(export_text(arbol.named_steps["clf"], feature_names=list(nombres_arbol), decimals=2))

fig, ax = plt.subplots(figsize=(14, 5.5))
plot_tree(arbol.named_steps["clf"], feature_names=nombres_arbol,
          class_names=["se queda", "abandona"], filled=True, rounded=True,
          fontsize=7, impurity=False, proportion=True, ax=ax)
ax.set_title("Tres preguntas bastan para ordenar la cartera de riesgo", fontsize=12)
plt.tight_layout()
plt.show()

| regla | quiénes | riesgo |
|---|---|---|
| Compró hace **88 días o menos** | la mitad de la cartera | Bajo. No se llama |
| Lleva **entre 89 y 263 días** sin comprar **y deja menos de 34,52 dólares de margen** | grupo pequeño | Alto (57,98 %) |
| Lleva **más de 263 días** sin comprar **y es cliente de menos de 747 días** | 148 clientes | **Muy alto (72,30 %)** |
| Lleva más de 263 días pero es cliente **antiguo** | grupo pequeño | Medio |

Eso es una hoja de ruta, no un modelo. El vendedor puede aplicarla con la libreta en la mano, y el
gerente puede discutirla: *«¿de verdad damos por perdido a un cliente antiguo que lleva un año sin
venir?»*. Esa discusión con una regresión logística es imposible.

Dos advertencias antes de enamorarse del árbol:

1. **Los cortes son frágiles.** 88,50 días no es un número con significado: es donde cayó el corte con
   estos datos. Con otra muestra sale 84 o 93. Se comunica como «unos tres meses», no como 88,5.
2. **El árbol sin podar memoriza**, y en la tabla de la sección 4 sale peor que no tener modelo.

### A4 · Precisión, sensibilidad y dinero a lo largo del umbral

Las dos curvas del barrido de la sección 6. La de la izquierda explica el intercambio entre precisión y sensibilidad; la de la derecha enseña que el óptimo es plano y que lo caro es dejar el umbral en 0,50 por inercia.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(barrido["umbral"], barrido["precisión"], marker="o", color="#4C72B0",
             label="precisión · de los que llamo, cuántos se iban")
axes[0].plot(barrido["umbral"], barrido["sensibilidad"], marker="o", color="#C44E52",
             label="sensibilidad · de los que se van, a cuántos llamo")
axes[0].axhline(COSTO_CONTACTO / (TASA_RETENCION * VALOR_CLIENTE), color="#55A868",
                linestyle="--", label="precisión mínima para que pague")
axes[0].set_xlabel("umbral de decisión")
axes[0].legend(fontsize=8)
axes[0].set_title("Subir el umbral compra precisión con sensibilidad", fontsize=11)

axes[1].plot(barrido["umbral"], barrido["retorno"], marker="o", color="#DD8452")
axes[1].axhline(0, color="grey", linewidth=1)
axes[1].axvline(mejor["umbral"], color="#55A868", linestyle="--",
                label=f"óptimo {mejor['umbral']:.2f}")
axes[1].set_xlabel("umbral de decisión")
axes[1].set_ylabel("retorno en dólares")
axes[1].legend(fontsize=9)
axes[1].set_title(f"El dinero se maximiza en {mejor['umbral']:.2f}, no en 0,50", fontsize=11)
plt.tight_layout()
plt.show()